<a href="https://colab.research.google.com/github/IA2526/trabajo-ia-equipo-4/blob/main/Trabajo_Evaluacion_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Grupo 4 - 20/05/2026 - IA - Trabajo IA - Evaluación

# Notebook 5: Evaluación y Optimización

En el Notebook 4 construimos el motor de búsqueda facial con VP-Tree y comprobamos su funcionamiento básico mediante un test de self-retrieval. En este notebook evaluaremos el sistema de forma más rigurosa:
- Mediremos la velocidad real del VP-Tree frente a una búsqueda lineal
- Verificaremos que el índice recupera correctamente las caras consultadas
- Calularemos el umbral de similitud óptimo.

## 0. Instalación de dependencias

In [ ]:
%%capture
!pip install vptree deepface tf-keras matplotlib scikit-learn

In [ ]:
import pickle
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import vptree
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import roc_curve, auc
%matplotlib inline

## 1. Cargamos el VP-Tree y los datos del Notebook 4

En el Notebook 4 guardamos `vptree.pkl` con los `paths` y `vecs_arr` ya preparados.
Lo descargamos directamente desde GitHub para no tener que reconstruir nada desde cero.

In [ ]:
%%capture
!wget -q VPTREE_URL

In [ ]:
with open('vptree.pkl', 'rb') as f:
    data = pickle.load(f)

paths = data['paths']
vecs_arr = data['vecs_arr']

print(f"Caras indexadas : {len(paths)}")
print(f"Dimensión : {vecs_arr.shape[1]}")

Caras indexadas : 38299
Dimensión : 128


Aunque `vptree.pkl` contiene los vectores y rutas, el objeto VP-Tree en sí no es serializable directamente con pickle por la función de distancia.

In [ ]:
indices = list(range(len(paths)))

def distancia(a, b):
    va = vecs_arr[int(a)] if isinstance(a, (int, np.integer)) else np.array(a, dtype=np.float32)
    vb = vecs_arr[int(b)] if isinstance(b, (int, np.integer)) else np.array(b, dtype=np.float32)
    return float(np.linalg.norm(va - vb))

print("Building VPTree...")
tree = vptree.VPTree(indices, distancia)
print("VP-Tree construido")

Building VPTree...
VP-Tree construido


## 2. Comparacion de velocidad: VP-Tree vs. Búsqueda Lineal

Para medir si existe una ventaja real con VP-Tree sobre una búsqueda lineal, tenemos que calcular la distancia euclidiana entre la consulta y todos los vectores del dataset. Comparamos los tiempos de respuesta de ambos métodos sobre una muestra de 200 consultas aleatorias, variando el número de vecinos `k` solicitados.

In [ ]:
def buscar_lineal(query_vec, k=5):
    dists   = np.linalg.norm(vecs_arr - query_vec, axis=1)
    top_idx = np.argsort(dists)[:k]
    return [(float(dists[i]), paths[i]) for i in top_idx]

def buscar_vptree(query_vec, k=5):
    results = tree.get_n_nearest_neighbors(query_vec, k)
    return [(float(d), paths[int(idx)]) for d, idx in results]

In [ ]:
random.seed(42)
muestra_idx = random.sample(range(len(paths)), 200)
muestra_vecs = [vecs_arr[i] for i in muestra_idx]

In [ ]:
t_lineal = {}
t_vptree = {}

for k in [1, 5, 10, 20, 50]:

    # Búsqueda lineal
    t0 = time.time()
    for qv in muestra_vecs:
        buscar_lineal(qv, k=k)
    t_lineal[k] = (time.time() - t0) / len(muestra_vecs) * 1000

    # Búsqueda VP-Tree
    t0 = time.time()
    for qv in muestra_vecs:
        buscar_vptree(qv, k=k)
    t_vptree[k] = (time.time() - t0) / len(muestra_vecs) * 1000

    print(f"{k:<2} {t_lineal[k]:<5.2f} {t_vptree[k]:<5.2f}")

1  14.09 110.17
5  13.55 160.13
10 13.54 163.17
20 15.39 165.01
50 16.41 174.48


### Interpretación de los resultados

El VP-Tree resulta más lento que la búsqueda lineal con NumPy. Esto no es un error si no que se debe a como estan implementadas las librerias. La librería `vptree` es una implementación en Python puro haciendo multiples llamadas Python por consulta, cada una con su overhead de intérprete. La búsqueda lineal con NumPy por otro lado ejecuta  una sola operación sobre toda la matriz, lo que con 38k vectores de 128 dimensiones tarda ~14 ms.

La ventaja teórica de un árbol métrico solo se refleja cuando la implementación es compilada. Para un dataset del tamaño del nuestro y con NumPy como competidor, la búsqueda lineal gana fair and square.

## 3. Test de Self-Retrieval

El test de self-retrieval consiste en consultar que una cara ya indexada sea el primer reultado al aparecer cuando se consulta sobre si misma.

Este test nos ayuda a validar que:

- Los embeddings se guardaron e indexaron correctamente.
- El VP-Tree devuelve el vecino más cercano correcto.

Ejecutamos el test sobre una muestra de 500 caras aleatorias.

In [ ]:
random.seed(7)
test_idx = random.sample(range(len(paths)), 500)

In [ ]:
aciertos= 0
fallos= []
for i in test_idx:
    qv  = vecs_arr[i]
    results = buscar_vptree(qv, k=1)
    dist, nombre = results[0]
    if nombre == paths[i]:
        aciertos += 1
    else:
        fallos.append((paths[i], nombre))

print(f"Self-retrieval accuracy : {aciertos}/{(len(test_idx))} ({100*aciertos/(len(test_idx)):.1f}%)")

if fallos:
    print(f"\nFallos ({len(fallos)}):")
    for orig, encontrado in fallos[:5]:
        print(f"  Consulta: {orig}  →  Resultado: {encontrado} ")

Self-retrieval accuracy : 472/500 (94.4%)

Fallos (28):
  Consulta: 2439_1.jpg  →  Resultado: 471779_1.jpg 
  Consulta: 1095_0.jpg  →  Resultado: 465916_0.jpg 
  Consulta: 1737_5.jpg  →  Resultado: 468442_5.jpg 
  Consulta: 2448_0.jpg  →  Resultado: 471704_0.jpg 
  Consulta: 220_2.jpg  →  Resultado: 472169_2.jpg 


## 4. Calibración del Umbral de Similitud

Primero definimos claramente que es el umbral de similitud.Es la distancia euclidiana máxima a partir de la cual consideramos que dos caras no son la misma persona.En nuestra demo de Gradio ,cuando el usuario sube una foto, el programa devuelve los k vecinos más cercanos del VP-Tree independientemente de lo lejos que estén. Sin umbral, siempre se devuelven 5 resultados aunque ninguno sea la persona buscada.

Para calibrar el umbral necesitamos dos distribuciones de distancias:

- Pares positivos: para cada cara consultamos su vecino más cercano en el índice (k=2, excluyendo la propia imagen). Con alta probabilidad son apariciones de la misma persona o caras muy similares.
- Pares negativos: pares aleatorios del dataset, que con 38.299 elementos son casi con certeza personas distintas.

El umbral óptimo es el valor de distancia que mejor separa ambas distribuciones.Vamos a calcularlo sobre una muestra de 5000 elementos.

Primeramente calculamos la distribucion positiva

In [ ]:
random.seed(42)
muestra = random.sample(range(len(paths)), 5000)

dist_pos = []
for i in muestra:
    results = buscar_vptree(vecs_arr[i], k=2)
    dist_pos.append(results[1][0])


Ahora calculamos la distribucion negativa

In [ ]:
random.seed(99)
pares_neg = []
while len(pares_neg) < 5000:
    a, b = random.randint(0, len(paths)-1), random.randint(0, len(paths)-1)
    if a != b:
        pares_neg.append((a, b))
dist_neg = [float(np.linalg.norm(vecs_arr[a] - vecs_arr[b])) for a, b in pares_neg]

In [ ]:
print(f"Positivos — media: {np.mean(dist_pos):.3f}  std: {np.std(dist_pos):.3f}")
print(f"Negativos — media: {np.mean(dist_neg):.3f}  std: {np.std(dist_neg):.3f}")

Positivos — media: 4.504  std: 3.051
Negativos — media: 12.515  std: 3.428


Ahora lo que haremos es barrer todos los posibles umbrales de distancia. Para ello calculamos para cada uno el true positive rate y false positive rate. La curva ROC resultante nos permite elegir el umbral óptimo, es decir, el punto donde más caras similares recuperamos cometiendo el menor número de errores posibles.

Construimos los  scores y labels

In [ ]:
scores = np.array([-d for d in dist_pos] + [-d for d in dist_neg])
labels = np.array([1] * len(dist_pos) + [0] * len(dist_neg))

In [ ]:
fpr, tpr, umbrales_roc = roc_curve(labels, scores)
umbral_opt = -umbrales_roc[np.argmax(tpr - fpr)]

print(f"Umbral óptimo : {umbral_opt:.4f}")
print(f"TPR           : {tpr[np.argmax(tpr - fpr)]:.4f}")
print(f"FPR           : {fpr[np.argmax(tpr - fpr)]:.4f}")

Umbral óptimo : 9.7189
TPR           : 0.9778
FPR           : 0.1690


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Curva ROC
axes[0].plot(fpr, tpr, lw=2, color='steelblue', label=f'ROC')
axes[0].scatter(fpr[np.argmax(tpr - fpr)], tpr[np.argmax(tpr - fpr)], s=80, zorder=5, color='tomato',
                label=f'Umbral óptimo d={umbral_opt:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curva ROC')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Distribuciones con umbral
bins = np.linspace(0, max(max(dist_pos), max(dist_neg)) * 1.05, 60)
axes[1].hist(dist_pos, bins=bins, alpha=0.6, label='Vecino más cercano', color='steelblue', edgecolor='white')
axes[1].hist(dist_neg, bins=bins, alpha=0.6, label='Pares aleatorios',   color='tomato',   edgecolor='white')
axes[1].axvline(umbral_opt, color='black', linestyle='--', lw=2,
                label=f'Umbral óptimo = {umbral_opt:.3f}')
axes[1].set_xlabel('Distancia euclidiana'); axes[1].set_ylabel('Número de pares')
axes[1].set_title('Umbral óptimo')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 1300x400 with 2 Axes>

### Limitación de la metodología

Esta calibración asume que el vecino más cercano de cada cara es una aparición de la misma persona. No tenemos forma reak de verificarlo,ya que esto implicaria que alguine anote manualmente las identidades de las caras recortadas del dataset.

## 5. Precisión, Recall y F1 en función del umbral

Con las distribuciones calculadas, evaluamos cómo varían precisión y recall al mover el umbral. El umbral que maximiza el F1-Score es el que mejor equilibra ambas métricas.

In [ ]:
umbrales_eval = np.linspace(min(min(dist_pos), min(dist_neg)), max(max(dist_pos), max(dist_neg)), 200)
precisiones, recalls, f1s = [], [], []

In [ ]:
for u in umbrales_eval:
    tp = sum(1 for d in dist_pos if d <= u)
    fp = sum(1 for d in dist_neg if d <= u)
    fn = sum(1 for d in dist_pos if d >  u)

    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else 0.0

    precisiones.append(prec)
    recalls.append(recall)
    f1s.append(f1)

best_umbral = umbrales_eval[int(np.argmax(f1s))]
best_prec   = precisiones[int(np.argmax(f1s))]
best_recall = recalls[int(np.argmax(f1s))]
best_f1     = f1s[int(np.argmax(f1s))]

In [ ]:
print(f"Umbral máx. F1 : {best_umbral:.4f}")
print(f"Precisión      : {best_prec:.4f}")
print(f"Recall         : {best_recall:.4f}")
print(f"F1-Score       : {best_f1:.4f}")

Umbral máx. F1 : 9.6550
Precisión      : 0.8541
Recall         : 0.9752
F1-Score       : 0.9106


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(umbrales_eval, precisiones, label='Precisión', color='steelblue')
plt.plot(umbrales_eval, recalls,     label='Recall',    color='tomato')
plt.plot(umbrales_eval, f1s,         label='F1-Score',  color='mediumseagreen', lw=2)
plt.axvline(best_umbral, color='black', linestyle='--', lw=1.5,
            label=f'Mejor umbral = {best_umbral:.3f}  (F1={best_f1:.3f})')
plt.xlabel('Umbral de distancia')
plt.ylabel('Valor de la métrica')
plt.title('Precisión, Recall y F1-Score en función del umbral')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 900x400 with 1 Axes>

## 6. Umbral de similitud en la demo Gradio

A partir del umbral óptimo calculado antes (d = 9.69), actualizamos `app.py` para filtrar resultados no relevantes. El cambio es mínimo:

```python
# ── 9. Búsqueda VP-Tree ────────────────────────────────────────────────────
UMBRAL = 9.69  

def _vp_query(emb_vec, k=5):
    query   = Vec(-1, np.array(emb_vec, dtype=np.float32))
    vecinos = vp.get_n_nearest_neighbors(query, k)
    return [(float(d), paths[v.idx]) for d, v in vecinos if d <= UMBRAL]
```

## 7. Conclusiones

En este notebook hemos evaluado el motor de búsqueda facial desde tres perspectivas. La comparativa de velocidad muestra que la búsqueda lineal con NumPy supera al VP-Tree en Python puro para un dataset de este tamaño, ya que la ventaja teórica del árbol métrico solo se materializa con implementaciones compiladas. El test de self-retrieval sobre 500 caras confirma que los embeddings se indexaron correctamente, recuperando  casi siempre la imagen consultada con distancia ≈ 0. Finalmente, el análisis ROC sobre 5.000 pares nos ha dado el umbral óptimo de similitud en d ≈ 9.69. Finalmente añadimos a la demo de Gradio este valor para filtrar resultados irrelevantes.